# Installation

## Granite Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch==2.7.1" "triton>=3.3.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# These are mamba kernels and we must have these for faster training
# Mamba kernels are for now supported only on torch==2.7.1. If you have newer torch versions, please wait 30 minutes for it to compile
!uv pip install --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2

## GPT OSS Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Model Creation

In [ ]:
from unsloth import FastLanguageModel

def create_granite_model(finetune: bool = False):

    if finetune:
        model_name = "avanishd/granite-4.0-h-tiny-finetuned-ground-news"
    else:
        model_name = "unsloth/granite-4.0-h-tiny"

    # We can try testing granite 4 small on a big GPU if micro isn't that good
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = 1024,   # Context length (longer = more memory) --> summaries are 500 words max, so this should be more than enough
        load_in_4bit = False,    # Load full precision for max accuracy
        load_in_8bit = False,
        full_finetuning = False, # Don't do this, LoRA gets just as good results
    )

    return model, tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastLanguageModel
import torch

def create_gpt_model(finetune: bool = False):

    if finetune:
      model_name = "avanishd/gpt-oss-20b-finetuned-ground-news"
    else:
      model_name = "unsloth/gpt-oss-20b" # Unsloth MXFP4 format (prefer over bnb-4bit since this is the format OpenAI post-trained the model w/)


    model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = model_name,
      dtype = None, # None for auto detection
      max_seq_length = 4096, # Need this to be higher b/c reasoning takes up token count
      load_in_4bit = False,
      full_finetuning = False,
      )
    
    return model, tokenizer

# Inference

In [ ]:
# Based on Sakhawat et al., 2026

zero_shot_system_prompt = """
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.
"""

# Handpicked from training set (if dataset changes, update these so there's no accidental data leakage)
few_shot_system_prompt = """
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.

RELEVANT EXAMPLES

HEADLINE: Video. Venezuela's Machado says she 'presented' her Nobel Peace Prize medal to Trump
ARTICLE SUMMARY: Video. Venezuelan opposition leader María Corina Machado has told reporters that she presented the medal for her Nobel Peace Prize to President Donald Trump at a private White House meeting on Thursday. 1 month ago · France
LABEL: 0.0

HEADLINE: Power Outages and Icy Cold by Winter Storm in the Usa
ARTICLE SUMMARY: Snow, cold, failures: A violent storm sweeps over the US. 190 million Americans struggle with the consequences. The authorities warn against long-lasting power failures.
LABEL: +2.0

HEADLINE: The u.s. Federal Reserve Curbed the Cut in Fees and Left Them at 3.75 per Cent.
ARTICLE SUMMARY: The US Federal Reserve noted that US inflation remains “something high” to what is expected and the fiscal perspective is still very limited.
LABEL: -2.0
"""

In [ ]:
# Load the dataset
from datasets import load_dataset

dataset = load_dataset('avanishd/ground-news-2026', split='test') # Only test set since this is just inference

README.md:   0%|          | 0.00/677 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/812k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/202k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/201k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4583 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/983 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/982 [00:00<?, ? examples/s]

In [ ]:
from transformers import TextStreamer
import polars as pl

In [ ]:
def granite_inference(inference_model, inference_tokenizer, few_shot: bool = False, output_csv_name: str = "granite_zero_shot.csv"):
  FastLanguageModel.for_inference(inference_model) # Enable native 2x faster inference

  # Conduct inference on the entire test set and save outputs to a csv file

  inference_df = pl.DataFrame(schema=
      {
          "headline": pl.String(),
          "summary": pl.String(),
          "bias": pl.String(),
          "predicted_bias": pl.String(),
      })

  for i in range(len(dataset)):
      headline = dataset[i]['headline']
      summary = dataset[i]['summary']
      true_bias = dataset[i]['bias']

      if not few_shot:
          default_system_prompt = zero_shot_system_prompt
      else:
          default_system_prompt = few_shot_system_prompt

      messages = [
          {"role": "system", "content": default_system_prompt},
          {"role": "user", "content": f"""
            HEADLINE: {headline}
            ARTICLE SUMMARY: {summary}

            Output ONLY the numeric bias score.
          """
          },
      ]

      inputs = inference_tokenizer.apply_chat_template(
          messages,
          tokenize = True,
          add_generation_prompt = True, # Must add for generation
          padding = True,
          return_tensors = "pt",
          return_dict = True,
      ).to("cuda")

      text_streamer = TextStreamer(inference_tokenizer, skip_prompt = False)

      output = inference_model.generate(**inputs,
                      streamer = text_streamer,
                      max_new_tokens = 8, # Even if they put category and not a number, they should not go more than this
                      use_cache = True,
                      do_sample = True,
                      temperature = 0.7, top_p = 0.8, top_k = 20,
      )

      predicted_bias = inference_tokenizer.decode(output[0], skip_special_tokens = True).strip()

      inference_df = pl.concat([inference_df, pl.DataFrame({
          "headline": [headline],
          "summary": [summary],
          "bias": [true_bias],
          "predicted_bias": [predicted_bias],
      })], how = "vertical")

  # Save output
  inference_df.write_csv(output_csv_name)


In [ ]:
def gpt_inference(inference_model, inference_tokenizer, few_shot: bool = False, reasoning_level: str = "low", output_csv_name: str = "granite_zero_shot.csv"):
  FastLanguageModel.for_inference(inference_model) # Enable native 2x faster inference

  # Conduct inference on the entire test set and save outputs to a csv file

  inference_df = pl.DataFrame(schema=
      {
          "headline": pl.String(),
          "summary": pl.String(),
          "bias": pl.String(),
          "predicted_bias": pl.String(),
      })

  for i in range(len(dataset)):
      headline = dataset[i]['headline']
      summary = dataset[i]['summary']
      true_bias = dataset[i]['bias']

      if not few_shot:
          default_system_prompt = zero_shot_system_prompt
      else:
          default_system_prompt = few_shot_system_prompt

      messages = [
          {"role": "developer", "content": default_system_prompt}, # Developer is like system level for Granite. See: https://developers.openai.com/cookbook/articles/openai-harmony
          {"role": "user", "content": f"""
            HEADLINE: {headline}
            ARTICLE SUMMARY: {summary}

            Output ONLY the numeric bias score.
          """
          },
      ]

      inputs = inference_tokenizer.apply_chat_template(
          messages,
          add_generation_prompt = True, # Must add for generation
          padding = True,
          return_tensors = "pt",
          return_dict = True,
          reasoning_effort = reasoning_level,
      ).to("cuda")

      match reasoning_level:
        case "low":
          max_new_tokens = 512
        case "medium":
          max_new_tokens = 1024
        case "high":
          max_new_tokens = 2048

      output = inference_model.generate(**inputs,
                      max_new_tokens = max_new_tokens,
                      streamer = TextStreamer(inference_tokenizer)
      )

      predicted_bias = inference_tokenizer.decode(output[0], skip_special_tokens = True).strip()

      inference_df = pl.concat([inference_df, pl.DataFrame({
          "headline": [headline],
          "summary": [summary],
          "bias": [true_bias],
          "predicted_bias": [predicted_bias],
      })], how = "vertical")

  # Save output
  inference_df.write_csv(output_csv_name)

## Granite

### Base Model

In [ ]:
model, tokenizer = create_granite_model()

==((====))==  Unsloth 2026.2.1: Fast Granitemoehybrid patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.07G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

#### Zero Shot

In [ ]:
granite_inference(model, tokenizer, few_shot = False, output_csv_name = "granite_zero_shot.csv")

Streaming output truncated to the last 5000 lines.
<|end_of_text|>
<|start_of_role|>user<|end_of_role|>
            HEADLINE: New research bolsters evidence that Tylenol doesn’t raise the risk of autism despite Trump’s claims
            ARTICLE SUMMARY: 1 month ago · Chicago, United States

            Output ONLY the numeric bias score.
          <|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>-1.5<|end_of_text|>
<|start_of_role|>system<|end_of_role|>
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.
<|end_of_text|>
<|start_of_role|>user<|end_of_role|>
            HEADLINE: At least 14 killed across US during winter storm, thousands of flights canceled again
            ARTICLE SUMMARY: At least 14 killed across US during winter storm, thousands of flights canceled again amid power outages. The U.S. Transportation Secretary says things wo

#### Few Shot

In [ ]:
granite_inference(model, tokenizer, few_shot = True, output_csv_name = "granite_few_shot.csv")

Streaming output truncated to the last 5000 lines.
            ARTICLE SUMMARY: The U.S. Department of Justice publishes 3 million additional documents related to the case of the deceased sexual criminal Jeffrey Epstein.

            Output ONLY the numeric bias score.
          <|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>+2.0<|end_of_text|>
<|start_of_role|>system<|end_of_role|>
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.

RELEVANT EXAMPLES

HEADLINE: Video. Venezuela's Machado says she 'presented' her Nobel Peace Prize medal to Trump
ARTICLE SUMMARY: Video. Venezuelan opposition leader María Corina Machado has told reporters that she presented the medal for her Nobel Peace Prize to President Donald Trump at a private White House meeting on Thursday. 1 month ago · France
LABEL: 0.0

HEADLINE: Power Outages and Icy Cold by Winter 

### Finetuned

In [ ]:
fine_tune_model, fine_tune_tokenizer = create_granite_model(finetune=True)

==((====))==  Unsloth 2026.2.1: Fast Granitemoehybrid patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.07G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
granite_inference(fine_tune_model, fine_tune_tokenizer, few_shot = False, output_csv_name = "granite_finetuned.csv")

Streaming output truncated to the last 5000 lines.
<|end_of_text|>
<|start_of_role|>user<|end_of_role|>
            HEADLINE: New research bolsters evidence that Tylenol doesn’t raise the risk of autism despite Trump’s claims
            ARTICLE SUMMARY: 1 month ago · Chicago, United States

            Output ONLY the numeric bias score.
          <|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>0.0<|end_of_text|>
<|start_of_role|>system<|end_of_role|>
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.
<|end_of_text|>
<|start_of_role|>user<|end_of_role|>
            HEADLINE: At least 14 killed across US during winter storm, thousands of flights canceled again
            ARTICLE SUMMARY: At least 14 killed across US during winter storm, thousands of flights canceled again amid power outages. The U.S. Transportation Secretary says things won

## GPT OSS

### Base Model

In [ ]:
model, tokenizer = create_gpt_model()

#### Zero Shot

In [ ]:
granite_inference(model, tokenizer, few_shot = False, reasoning_level = "low", output_csv_name = "gpt_zero_shot_low.csv")

In [ ]:
granite_inference(model, tokenizer, few_shot = False, reasoning_level = "medium", output_csv_name = "gpt_zero_shot_medium.csv")

In [ ]:
granite_inference(model, tokenizer, few_shot = False, reasoning_level = "high", output_csv_name = "gpt_zero_shot_high.csv")

#### Few Shot

In [ ]:
granite_inference(model, tokenizer, few_shot = True, reasoning_level = "low", output_csv_name = "gpt_few_shot_low.csv")

In [ ]:
granite_inference(model, tokenizer, few_shot = True, reasoning_level = "medium", output_csv_name = "gpt_few_shot_medium.csv")

In [ ]:
granite_inference(model, tokenizer, few_shot = True, reasoning_level = "high", output_csv_name = "gpt_few_shot_high.csv")

#### Fine Tuned